In [1]:
import pandas as pd
from GNN.architecture import GNNTrainer
import torch
import pickle
from sklearn.metrics import ndcg_score
import numpy as np
import random
from numpy import unravel_index

## LOAD THE DATASET

In [2]:
train_data = pd.read_csv("data/movielens/train.csv", index_col=0)
val_data = pd.read_csv("data/movielens/test.csv", index_col=0)
movie_features = pd.read_csv("data/movielens/movie_features.csv", index_col=0)
train_pair_data = np.loadtxt("data/movielens/train_pair.csv", dtype=np.int64, delimiter=",")
val_pair_data = np.loadtxt("data/movielens/val_pair.csv", dtype=np.int64, delimiter=",")


/tmp/ipykernel_2043270/1152477808.py:4: DeprecationWarning: loadtxt(): Parsing an integer via a float is deprecated.  To avoid this warning, you can:
    * make sure the original data is stored as integers.
    * use the `converters=` keyword argument.  If you only use
      NumPy 1.23 or later, `converters=float` will normally work.
    * Use `np.loadtxt(...).astype(np.int64)` parsing the file as
      floating point and then convert it.  (On all NumPy versions.)
  (Deprecated NumPy 1.23)
  train_pair_data = np.loadtxt("data/movielens/train_pair.csv", dtype=np.int64, delimiter=",")
/tmp/ipykernel_2043270/1152477808.py:5: DeprecationWarning: loadtxt(): Parsing an integer via a float is deprecated.  To avoid this warning, you can:
    * make sure the original data is stored as integers.
    * use the `converters=` keyword argument.  If you only use
      NumPy 1.23 or later, `converters=float` will normally work.
    * Use `np.loadtxt(...).astype(np.int64)` parsing the file as
      flo

## NEGATIVE SAMPLING - FOR 1 POSITVE SAMPLE THERE WILL BE 100 NEGATIVE SAMPLES

In [3]:
all_users = set(train_data['userId'].unique())
all_items = set(train_data['productId'].unique())

In [4]:
users_negative_samples = {}
for i,u in enumerate(all_users):
    user_u_positive_items = set(train_data[train_data['userId']==u]['productId'].unique())
    user_u_negative_items = all_items.difference(user_u_positive_items)
    users_negative_samples[u] = user_u_negative_items

In [5]:
all_pairs = []
for i ,(userId, productId_pos, _, _) in enumerate(train_data.values):
    sample_negative_items = random.sample(list(users_negative_samples[userId]), 100)
    pair_data_u = [[userId, productId_pos, userId, productId_neg] for productId_neg in sample_negative_items]
    all_pairs.extend(pair_data_u)
    if (i+1) % 1000 == 0:
        print(f"{i+1} records complete")

1000 records complete
2000 records complete
3000 records complete
4000 records complete
5000 records complete
6000 records complete
7000 records complete
8000 records complete
9000 records complete
10000 records complete
11000 records complete
12000 records complete
13000 records complete
14000 records complete
15000 records complete
16000 records complete
17000 records complete
18000 records complete
19000 records complete
20000 records complete
21000 records complete
22000 records complete
23000 records complete
24000 records complete
25000 records complete
26000 records complete
27000 records complete
28000 records complete
29000 records complete
30000 records complete
31000 records complete
32000 records complete
33000 records complete
34000 records complete
35000 records complete
36000 records complete
37000 records complete
38000 records complete
39000 records complete
40000 records complete
41000 records complete
42000 records complete
43000 records complete
44000 records comple

In [6]:
# SAVE PAIR DATA IN CSV FORMAT - userId, productId_pos, userId, productId_neg
all_pairs_np = np.array(all_pairs)

In [ ]:
np.savetxt("data/movielens/train_pair.csv", all_pairs_np, delimiter=",")

In [19]:
# Validation pair
all_users = set(val_data['userId'].unique())
all_items = set(val_data['productId'].unique())

In [20]:
users_negative_samples = {}
for i,u in enumerate(all_users):
    user_u_positive_items = set(val_data[val_data['userId']==u]['productId'].unique())
    user_u_negative_items = all_items.difference(user_u_positive_items)
    users_negative_samples[u] = user_u_negative_items

In [14]:
all_pairs = []
for i ,(userId, productId_pos, _, _) in enumerate(val_data.values):
    sample_negative_items = random.sample(list(users_negative_samples[userId]), 20)
    pair_data_u = [[userId, productId_pos, userId, productId_neg] for productId_neg in sample_negative_items]
    all_pairs.extend(pair_data_u)
    if (i+1) % 1000 == 0:
        print(f"{i+1} records complete")

1000 records complete
2000 records complete
3000 records complete
4000 records complete
5000 records complete
6000 records complete
7000 records complete
8000 records complete
9000 records complete
10000 records complete
11000 records complete
12000 records complete
13000 records complete
14000 records complete
15000 records complete
16000 records complete
17000 records complete
18000 records complete
19000 records complete
20000 records complete
21000 records complete
22000 records complete
23000 records complete
24000 records complete
25000 records complete
26000 records complete
27000 records complete
28000 records complete
29000 records complete
30000 records complete
31000 records complete
32000 records complete
33000 records complete
34000 records complete
35000 records complete
36000 records complete
37000 records complete
38000 records complete
39000 records complete
40000 records complete
41000 records complete
42000 records complete
43000 records complete
44000 records comple

In [26]:
all_pairs_np = np.array(all_pairs)

In [27]:
np.savetxt("data/movielens/val_pair.csv", all_pairs_np, delimiter=",")

## TRAINER - TRAIN FOR 10 EPOCHS

In [3]:
trainer = GNNTrainer(train_data, movie_features, train_pair_data, val_pair_data)

/home/arnabroy/Workspace/recommendation_system/GNN/architecture.py:187: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_movie_features.sort_index(inplace=True)


In [5]:
trainer.train_data

HeteroData(
  user={ x=[7964] },
  product={ x=[9725] },
  product_feature={ x=[9725, 20] },
  (user, rates, product)={ edge_index=[2, 698113] },
  (product, rated_by, user)={ edge_index=[2, 698113] }
)

In [ ]:
trainer.train(epochs=10, lr=0.0005, decay=10**(-4))

## MEASURE PERFORMANCE - OFFLINE NDCG 24.9%

In [38]:
val_df = pd.read_csv("data/movielens/test.csv", index_col=0)
val_df.head(3)

,userId,productId,rating,timestamp
20011489,195660,541,4.5,2022-10-17 16:31:56
20011492,195660,750,5.0,2022-10-17 16:32:12
20011523,195660,3504,5.0,2022-10-17 16:32:25


In [39]:
with open("models/GNN/model_2.pt", "rb") as f:
    model = torch.load(f)
    f.close()
with open("models/GNN/encoder.pkl", "rb") as f:
    enc = pickle.load(f)
    f.close()
model.to('cpu')
model.final_user_emb = model.final_user_emb.to('cpu')
model.final_item_emb = model.final_item_emb.to('cpu')

/tmp/ipykernel_2038709/4174936051.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(f)


In [40]:
val_df['userId'] = val_df['userId'].map(lambda x: enc.user_id[x])
val_df['productId'] = val_df['productId'].map(lambda x: enc.item_id[x])
val_df.head(5)

,userId,productId,rating,timestamp
20011489,7959,44,4.5,2022-10-17 16:31:56
20011492,7959,867,5.0,2022-10-17 16:32:12
20011523,7959,2869,5.0,2022-10-17 16:32:25
27951262,7962,1386,4.0,2022-10-17 16:32:39
27951145,7962,1695,4.0,2022-10-17 16:32:44


In [41]:
gt_df = val_df.pivot(index='userId', columns='productId', values='rating').fillna(0.0)

In [ ]:
all_users = gt_df.index.values
all_items = gt_df.columns.values

In [42]:
pred = torch.full(size=(gt_df.shape[0], gt_df.shape[1]),fill_value=0.0)

In [43]:
users_xx, items_yy = torch.meshgrid(torch.tensor(all_users) , torch.tensor(all_items))

In [44]:
for i in range(pred.shape[0]):
    users = users_xx[i]
    items = items_yy[i]
    ratings = model.predict(users, items)
    pred[i] = ratings

In [45]:
avg_ndcg = ndcg_score(gt_df.values, pred.detach().numpy())
avg_ndcg_20 = ndcg_score(gt_df.values, pred.detach().numpy(), k=20)

In [46]:
print(f"Offline NDCG average is {avg_ndcg*100: 0.2f}%")
print(f"Offline NDCG@20 is {avg_ndcg_20*100: 0.2f}%")


Offline NDCG average is  29.21%
Offline NDCG@20 is  2.66%


## EXPLORING REPRESENTATIONS

In [16]:
movies = pd.read_csv('data/movielens/movies.csv')
ratings = pd.read_csv('data/movielens/train.csv', index_col=0)

In [17]:
with open("models/GNN/best.pt", "rb") as f:
    model = torch.load(f)
    f.close()
with open("models/GNN/encoder.pkl", "rb") as f:
    enc = pickle.load(f)
    f.close()
model.to('cpu')
model.final_user_emb = model.final_user_emb.detach().cpu().numpy()
model.final_item_emb = model.final_item_emb.detach().cpu().numpy()

/tmp/ipykernel_2033931/3233660656.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(f)


In [21]:
# MOST SIMILAR MOVIE PAIRS
movie_sim = model.final_item_emb@model.final_item_emb.T
np.fill_diagonal(movie_sim, -1)
max_pos = unravel_index(movie_sim.argmax(), movie_sim.shape)
movie_1, movie_2 = enc.id_item[max_pos[0]], enc.id_item[max_pos[1]]
print("Most similar movie pairs")
movies[movies['movieId'].isin([movie_1, movie_2])]

Most similar movie pairs


,movieId,title,genres
3858,3961,Ghoulies (1985),Horror
6289,6409,Bend of the River (1952),Western


In [27]:
# Similar movie of any random movie
movie = enc.id_item[2000]
print(movies[movies['movieId'].isin([movie])])
movies[movies['movieId'].isin([movie])]
top_10 = movie_sim[2000].argsort()[::-1][:10]
top_10 = [enc.id_item[x] for x in top_10]
print('\nTop 10 similar movies of the above movie based on the viewing pattern')
movies[movies['movieId'].isin(top_10)]

       movieId                                        title  \
61208   202517  Kung Fu Panda: Secrets of the Scroll (2016)   

                                           genres  
61208  Action|Adventure|Animation|Children|Comedy  

Top 10 similar movies of the above movie based on the viewing pattern


,movieId,title,genres
12223,58559,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX
12326,59315,Iron Man (2008),Action|Adventure|Sci-Fi
13363,68954,Up (2009),Adventure|Animation|Children|Drama
21951,112852,Guardians of the Galaxy (2014),Action|Adventure|Sci-Fi
22593,115617,Big Hero 6 (2014),Action|Animation|Comedy
25115,122904,Deadpool (2016),Action|Adventure|Comedy|Sci-Fi
25119,122912,Avengers: Infinity War - Part I (2018),Action|Adventure|Sci-Fi
25120,122914,Avengers: Infinity War - Part II (2019),Action|Adventure|Sci-Fi
25121,122916,Thor: Ragnarok (2017),Action|Adventure|Sci-Fi
54254,187593,Deadpool 2 (2018),Action|Comedy|Sci-Fi


In [24]:
movie_sim[np.where(movie_sim < 0)].shape

(9725,)

In [26]:
user = enc.id_user[1048]
print(f"Top 10 rated movies by user {user} ")
df = ratings[ratings['userId'] == user].sort_values(by='rating', ascending=False)
df['movie'] = df['productId'].map(lambda x: movies[movies['movieId']==x]['title'].values[0])
df[['userId', 'movie', 'rating']].iloc[:10]

Top 10 rated movies by user 61485 


,userId,movie,rating
6303770,61485,The Bee Gees: How Can You Mend a Broken Heart ...,4.0
6302397,61485,Mad Max (1979),4.0
6302398,61485,"Road Warrior, The (Mad Max 2) (1981)",4.0
6303526,61485,Trick 'r Treat (2007),4.0
6303777,61485,Spider-Man: No Way Home (2021),4.0
6301861,61485,Conan the Barbarian (1982),4.0
6303778,61485,The Batman (2022),4.0
6303779,61485,Scream (2022),4.0
6301803,61485,Dracula (Bram Stoker's Dracula) (1992),4.0
6302431,61485,Pumpkinhead (1988),4.0


In [ ]:
with open("models/GNN/best.pt", "rb") as f:
    model = torch.load(f)
    f.close()

/tmp/ipykernel_1997534/3712743250.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(f)


In [25]:
movie_sim

array([[-1.      , 34.80675 , 34.83048 , ..., 28.088045, 24.817478,
        22.38975 ],
       [34.80675 , -1.      , 32.72488 , ..., 30.040165, 26.79695 ,
        26.324774],
       [34.83048 , 32.72488 , -1.      , ..., 26.44548 , 23.2055  ,
        23.001467],
       ...,
       [28.088045, 30.040165, 26.44548 , ..., -1.      , 21.542809,
        21.130142],
       [24.817478, 26.79695 , 23.2055  , ..., 21.542809, -1.      ,
        19.894344],
       [22.38975 , 26.324774, 23.001467, ..., 21.130142, 19.894344,
        -1.      ]], dtype=float32)

In [29]:
model.predict(
    torch.tensor([100, 200], device='cuda:1'),
    torch.tensor([200, 300], device='cuda:1') 
)

TypeError: can't convert cuda:1 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [34]:
next(model.parameters()).device

device(type='cpu')